# Netflix Titles — Data Cleaning

This notebook practices core data-cleaning skills on the Netflix Titles dataset:

1. **Part 1 – Quantifying missing data**
2. **Part 2 – Comparing cleaning strategies (imputing vs. dropping rows)**
3. **Part 3 – Processing datetime data**

In [1]:
import pandas as pd

# Load the dataset into a DataFrame named netflix_titles
netflix_titles = pd.read_csv("netflix_titles-1.csv")

# Display the initial shape of the dataset
print(f"Initial shape (rows, columns): {netflix_titles.shape}")

Initial shape (rows, columns): (8807, 12)


## Part 1 – Quantifying Missing Data

### Missing Value Audit

Absolute number of missing data points per column, using `.isnull().sum()`:

In [2]:
# Absolute number of missing values per column
netflix_titles.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

### Impact Scaling

Percentage of missing data for each column:

In [3]:
# Percentage of missing data per column
(netflix_titles.isnull().sum() / len(netflix_titles) * 100).round(2)

show_id          0.00
type             0.00
title            0.00
director        29.91
cast             9.37
country          9.44
date_added       0.11
release_year     0.00
rating           0.05
duration         0.03
listed_in        0.00
description      0.00
dtype: float64

#### Which columns suffer from the worst data gaps?

The **`director`** column has by far the worst data gap: **2,634 missing values (~29.91%)** — almost 1 in 3 titles has no director recorded. The second tier of problems is **`country`** (831 missing, ~9.44%) and **`cast`** (825 missing, ~9.37%), each missing for roughly 1 in 10 titles. The remaining gaps are trivial: `date_added` (10, ~0.11%), `rating` (4, ~0.05%) and `duration` (3, ~0.03%). The other columns (`show_id`, `type`, `title`, `release_year`, `listed_in`, `description`) are fully complete.

## Part 2 – Cleaning Strategies (Dropping vs. Imputing)

Data analysts must choose between deleting missing records or filling them with placeholders. Run both tests below to compare the strategies.

### Strategy A (Imputation)

The `director` column has numerous missing values. Replace every `NaN` in `director` with the string placeholder `"Unknown"` and verify the missing count drops to 0:

In [4]:
# Strategy A: impute missing directors with the placeholder "Unknown"
netflix_titles["director"] = netflix_titles["director"].fillna("Unknown")

# Verify that the missing count for director has dropped to 0
print("Missing values in 'director' after imputation:", netflix_titles["director"].isnull().sum())

Missing values in 'director' after imputation: 0


### Strategy B (Row Dropping)

Create a separate experimental DataFrame and remove any rows where `director` **or** `cast` contain missing data using `.dropna()`. (Since Strategy A already filled every missing `director` with `"Unknown"`, only rows missing a `cast` are affected.)

In [5]:
# Strategy B: separate experimental DataFrame (the main DataFrame stays intact)
netflix_dropna = netflix_titles.copy()

# Remove any rows where director or cast contain missing data
netflix_dropna = netflix_dropna.dropna(subset=["director", "cast"])

# Final shape after Strategy B
print(f"Shape after Strategy B (rows, columns): {netflix_dropna.shape}")

Shape after Strategy B (rows, columns): (7982, 12)


#### Reflection — how much data did Strategy B lose?

Strategy B shrank the dataset from **8,807 rows to 7,982 rows**:

**8,807 - 7,982 = 825 rows lost (~9.37% of the dataset)** — exactly the share of titles that are missing a `cast` list (the `director` gaps had already been imputed by Strategy A, so no rows were lost for `director`).

**A scenario where losing this much data could ruin an analysis:** imagine studying *which actors appear most frequently across Netflix titles*. All 825 dropped rows are titles whose entire cast is unknown — deleting them does not fix the missing information, it silently deletes exactly the population we need to study, so any "most featured actor" ranking would be biased toward titles with complete credits. The damage gets worse when the data is sliced: if we filtered to a smaller segment (e.g., only TV shows from one country, or titles added in a single year), the ~9% loss is amplified — it could wipe out entire subgroups and leave too few samples for statistically meaningful conclusions. This is why imputation (Strategy A), which preserves all 8,807 rows, is usually preferred for a column-level gap like `director`.

## Part 3 – Processing Datetime Data

### Data Type Verification

Check the data type of the `date_added` column using `.dtypes`. Notice that it is stored as a generic text string (`object` in older pandas versions; shown as `str` in pandas 3) rather than a true datetime type:

In [6]:
# Check the data type of the date_added column
print(netflix_titles["date_added"].dtypes)

str


### Datetime Conversion

Convert `date_added` from a text string into a true datetime type using `pd.to_datetime()`. The dates follow the pattern `"September 25, 2021"`, and a handful of entries contain a stray leading space, so we strip whitespace first and pass the explicit format `"%B %d, %Y"`:

In [7]:
# Convert date_added from a text string into a true datetime type
netflix_titles["date_added"] = pd.to_datetime(
    netflix_titles["date_added"].str.strip(), format="%B %d, %Y"
)

# Verify the transformation using .info()
netflix_titles.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   str           
 1   type          8807 non-null   str           
 2   title         8807 non-null   str           
 3   director      8807 non-null   str           
 4   cast          7982 non-null   str           
 5   country       7976 non-null   str           
 6   date_added    8797 non-null   datetime64[us]
 7   release_year  8807 non-null   int64         
 8   rating        8803 non-null   str           
 9   duration      8804 non-null   str           
 10  listed_in     8807 non-null   str           
 11  description   8807 non-null   str           
dtypes: datetime64[us](1), int64(1), str(10)
memory usage: 825.8 KB


### Feature Extraction

Use the `.dt` accessor to extract the numeric **day of the month** from the new datetime column and store it in a new column named `day_added`:

In [8]:
# Extract the numeric day of the month into a new column
netflix_titles["day_added"] = netflix_titles["date_added"].dt.day

# Display the first 5 rows to verify the change
netflix_titles.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,day_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",25.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",24.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,24.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,NaN,NaN,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",24.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,24.0
